# 📚 Lecture 11 Activity Notebook: Topic Modeling
## AA640: Data Analytics and Text Mining
### Bryant University | Prof. Gianluca Brero

---

**IMPORTANT:** *Before starting, save a copy to your Drive via `File > "Save a copy in Drive"`*

### 🎯 Estimated Time: 90 Minutes

#### 💻 This is a REMOTE lecture
Since we're working remotely today, this notebook has **extra-detailed explanations** in every section. Read the markdown cells carefully — they replace the in-person lecture walkthrough.

#### In-Class Schedule

| Time | Clock | Block |
|------|-------|-------|
| 30 min | 6:30pm | Lecture: What is Topic Modeling? |
| 20 min | 7:00pm | **Activity 1** — Train LDA on Newspaper Articles |
| 30 min | 7:20pm | Lecture: Tuning & Interpretation |
| 30 min | 7:50pm | Break + Quiz |
| 20 min | 8:20pm | **Activity 2** — Tune Model, Compare Newspapers |
| 30 min | 8:40pm | Open time — questions, finish notebook |

#### After-Class Activities (~30 min)
*Complete these on your own after lecture.*

| Activity | Topic | Time |
|----------|-------|------|
| Activity 3 | Single-Document Topic Analysis | ~15 min |
| Activity 4 | Topic Coverage Bar Chart | ~15 min |

### ⚙️ How to Use This Notebook
1. **Read** each section carefully — especially the markdown explanations (this is a remote lecture!)
2. **Run** the example cells first to see how things work
3. **Complete** the activities marked with 📝
4. **Check** your answers against the expected output

> 💡 **Tip:** If you get stuck, re-read the demo cell right above the activity — the pattern is always there!

## Setup: Install and Import Libraries

We need a few libraries for today:
- **gensim** — the main library for topic modeling (LDA)
- **pyLDAvis** — creates interactive topic visualizations
- **datasets** — lets us download the American Stories dataset from Hugging Face

Run the cell below to install them. This may take a minute.

In [ ]:
!pip install -q gensim pyLDAvis "datasets<4.0"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("Setup complete!")

---
# 📰 Loading the Dataset: American Stories (1958)

We'll use the **American Stories** dataset from Hugging Face — a collection of digitized newspaper articles from the United States spanning 1770–1964.

**What makes this dataset interesting:**
- Articles were scanned from microfilm and converted to text using OCR (optical character recognition)
- This means the text has some quirks: missing letters, garbled words, and artifacts from the scanning process
- Each article includes metadata: newspaper name, date, page, headline

We'll load articles from **1958** and sample 1,000 of them to keep things fast.

> 💡 **Note:** The first time you run this, it downloads ~200MB of data. Be patient!

In [ ]:
from datasets import load_dataset

# Download data for the year 1958
dataset = load_dataset("dell-research-harvard/AmericanStories",
                       "subset_years",
                       year_list=["1958"],
                       trust_remote_code=True)
print(dataset)

In [ ]:
# Convert to pandas and sample 1,000 articles
df_all = dataset['1958'].to_pandas()
df = df_all.sample(n=1000, random_state=33).copy()
df = df.reset_index(drop=True)

print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print()
print("Columns:", list(df.columns))
print()
df.head()

### Understanding the Columns

| Column | What it contains |
|--------|------------------|
| `article_id` | Unique ID for each article |
| `newspaper_name` | Name of the newspaper |
| `edition` | Edition number |
| `date` | Publication date (YYYY-MM-DD) |
| `page` | Page number in the printed paper |
| `headline` | Article headline |
| `byline` | Author or wire service |
| `article` | **Full text** — this is what we'll analyze |

### Quick Look at the Text

Let's look at one article to see what we're working with. Notice the OCR artifacts — some words are garbled because the scanner misread the microfilm.

In [ ]:
# Look at the first article (raw text)
print(df.loc[0, 'article'][:500])
print('...')

### Clean Up Newlines

The original text has `\n` (newline characters) from the newspaper's column layout. We'll join hyphenated words split across lines and replace remaining newlines with spaces.

In [ ]:
# Fix hyphenated line breaks and replace newlines with spaces
df['article'] = df['article'].str.replace('-\n', '').str.replace('\n', ' ')

print("Cleaned text (first 300 chars):")
print(df.loc[0, 'article'][:300])

---
# 📌 IN-CLASS ACTIVITIES

Complete these sections during class time. Each activity has demo cells above it — run the demos first, then work on the 📝 activity.

---

## Part 1: Preprocessing Text with Gensim

Before we can do topic modeling, we need to **clean and tokenize** the text. In previous lectures, we wrote custom cleaning functions (lowercasing, removing punctuation, filtering stopwords). Gensim has a built-in function that does all of this in one step.

**`preprocess_string()`** does the following automatically:
- Lowercases all text
- Removes punctuation and symbols
- Strips out numbers
- Filters out common stopwords ("the", "is", "and", etc.)
- Applies stemming ("running" → "run", "president" → "presid")

The result is a **list of cleaned tokens** ready for analysis.

### Demo: Preprocessing a Single Article

In [ ]:
from gensim.parsing.preprocessing import preprocess_string

# Preprocess a single article
sample_tokens = preprocess_string(df.loc[0, 'article'])

print("Original text (first 200 chars):")
print(df.loc[0, 'article'][:200])
print()
print("After preprocessing (first 20 tokens):")
print(sample_tokens[:20])

### Demo: Tokenize All Articles

We apply `preprocess_string` to every article in our DataFrame. The `.apply()` method runs the function on each row.

In [ ]:
# Apply preprocessing to every article
df['tokens'] = df['article'].apply(preprocess_string)

print(f"Tokenized {len(df)} articles.")
print()
print("Example tokens for article 0:")
print(df.loc[0, 'tokens'][:15])

### Demo: Most Common Words

Before building a topic model, it's useful to see which words appear most frequently. We use Python's `Counter` to count all tokens across all articles.

In [ ]:
# Count word frequencies across all articles
all_tokens = [token for tokens in df['tokens'] for token in tokens]
counts = Counter(all_tokens)

print(f"Total unique words: {len(counts)}")
print()
print("Top 10 most common words:")
for word, count in counts.most_common(10):
    print(f"  {word:15s} {count}")

## Part 2: Building the Dictionary and Corpus

LDA doesn't read raw text — it needs a special format called a **Bag of Words (BoW)**. Getting there requires two objects:

1. **Dictionary** — a mapping from every unique word to an ID number (like a codebook)
2. **Corpus** — each document converted to a list of (word_id, count) pairs

**Why filter extremes?** Words that appear in almost every document (like "said" or "new") don't help distinguish topics. Words that appear only once are probably typos or OCR errors. We keep the middle ground.

### Demo: Create Dictionary and Corpus

In [ ]:
from gensim.corpora import Dictionary

# Convert token column to a list of lists
texts = df['tokens'].tolist()

# Build the dictionary
dictionary = Dictionary(texts)
print(f"Dictionary before filtering: {len(dictionary)} unique words")

# Filter out very rare and very common words
dictionary.filter_extremes(no_below=5, no_above=0.90, keep_n=10000)
dictionary.compactify()
print(f"Dictionary after filtering:  {len(dictionary)} unique words")

# Convert documents to Bag of Words
corpus = [dictionary.doc2bow(text) for text in texts]

print(f"\nCorpus has {len(corpus)} documents.")
print(f"\nFirst document as BoW (first 5 entries):")
print(corpus[0][:5])

### Demo: Reading the Bag of Words

The BoW representation looks like `[(42, 3), (87, 2)]` which means word #42 appeared 3 times, word #87 appeared 2 times, etc. Let's translate back to words:

In [ ]:
# Show BoW as readable words for the first document
readable = [(dictionary[word_id], freq) for word_id, freq in corpus[0]]
print("First document as readable BoW (first 10 entries):")
for word, freq in readable[:10]:
    print(f"  {word:15s} appears {freq} time(s)")

## Part 3: Training an LDA Model

Now we have our Dictionary and Corpus — we're ready to train LDA!

**What LDA does:** It reads all the Bag of Words documents and tries to discover groups of words that tend to appear together. Each group becomes a "topic."

**Key parameters:**
- `num_topics` — how many topics to discover (you choose this)
- `passes` — how many times the algorithm scans through all documents (more = better but slower)
- `corpus` — the Bag of Words representation
- `id2word` — the dictionary so LDA can translate IDs back to words

### Demo: Train LDA with 5 Topics

In [ ]:
from gensim.models.ldamodel import LdaModel

# Train LDA model
num_topics = 5
passes = 15

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary.id2token,
    num_topics=num_topics,
    passes=passes,
    eval_every=None)

print("LDA model trained!")
print(f"Number of topics: {lda_model.num_topics}")

### Demo: View the Topics

Each topic is a list of words with weights. The higher the weight, the more important that word is to the topic. Your job is to look at the top words and give the topic a human-readable label.

In [ ]:
# Print all topics
topics = lda_model.print_topics(num_words=10)
for topic_id, topic_words in topics:
    print(f"Topic {topic_id}: {topic_words}")
    print()

---
## Activity 1: Train Your Own LDA Model (~20 min)

*You're a data analyst at a digital archives company. Your manager says: "We just digitized thousands of 1958 newspaper articles. The historians want to know: what were the main themes in these papers? Load a sample, train a model, and show me the topics."*

The demos above walked you through every step. Now put it all together yourself!

**Important for remote learners:** If you get stuck on any step, scroll up and re-read the corresponding demo. The code pattern is the same — you just need to combine the steps.

### 📝 Activity 1a: Train LDA with 8 Topics

Train a **new** LDA model with **8 topics** instead of 5. Use 15 passes.

Steps:
1. Create a new `LdaModel` with `num_topics=8`
2. Use the same `corpus`, `dictionary.id2token`, and `passes=15`
3. Print the topics

*Hint: Copy the demo code and change `num_topics` to 8.*

In [ ]:
# Activity 1a: Train LDA with 8 topics
# Expected output: 8 numbered topics, each printed with its top 10 weighted words.
# Reuse `corpus` and `dictionary` from the demos. Store the model in `lda_model_8`.

lda_model_8 =LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=8,
    random_state=42,
    passes=10
)
for i, topic in lda_model_8.print_topics(num_topics=8, num_words=10):
    print(f"Topic {i}: {topic}")
    print()


### 📝 Activity 1b: Label the Topics

Look at the top words in each of the 8 topics from Activity 1a. For each topic, write a short human-readable label that describes what the topic is about.

For example, if a topic has words like `game, team, score, player, win`, you might label it **"Sports"**.

Fill in the labels below:

In [ ]:
# Activity 1b: Read the top words of each topic above, then assign a short label.
# Expected output: 8 lines like "Topic 0 [<your label>]: <top 6 words>"

# Fill in your labels (one per topic):
topic_labels = {
    0: "...",
    1: "...",
    2: "...",
    3: "...",
    4: "...",
    5: "...",
    6: "...",
    7: "...",
}

# TODO: write a loop that prints each topic with its label and top 6 words.
for i, topic in lda_model_8.print_topics(num_topics=8, num_words=6):
    print(f"Topic {i}: {topic_labels[i]}: {topic}")
    print()


### 📝 Activity 1c: Examine a Single Document's Topic Mix

Pick a document and find out what mix of topics LDA assigned to it. Remember: every document is a **mixture** of topics.

Steps:
1. Choose a document index (try `doc_idx = 10`)
2. Use `lda_model_8.get_document_topics(corpus[doc_idx])` to get its topic breakdown
3. Print the original article text and the topic percentages

*Hint: The reference notebook shows this with `document_idx = 277`.*

In [ ]:
# Activity 1c: Inspect the topic mixture of a single article.
# Expected output: a list of (topic, percentage) pairs for one document, plus
# the first ~300 characters of that article so you can sanity-check the mix.

# Pick any document index from 0 to len(corpus)-1
doc_idx = 10

# TODO:
#  1. Use lda_model_8.get_document_topics(corpus[doc_idx]) to get the breakdown
#  2. Loop over the (topic_id, proportion) pairs and print them with their labels
#  3. Print the original article: df.loc[doc_idx, 'article'][:300]
topic_dist = lda_model_8.get_document_topics(corpus[doc_idx])
print("Topic mixture for document:\n")
for topic_id, proportion in topic_dist:
    print(f"Topic {topic_id}: {proportion:.3f}")
    print("\nArticle preview:\n")
print(df.loc[doc_idx, 'article'][:300])



---
## Activity 2: Tune and Compare (~20 min)

*Your manager is impressed and asks: "How do I know 8 topics is the right number? And do all newspapers cover the same topics, or are there differences?"*

**Important for remote learners:** The coherence computation below takes 1–2 minutes to run because it trains multiple LDA models. Be patient and read the explanations while you wait.

### Demo: Coherence Score

The **coherence score** measures how interpretable a topic model's topics are. It checks whether the top words in each topic actually tend to appear together in documents. Higher = better.

We compute the coherence score for our 5-topic model:

In [ ]:
from gensim.models import CoherenceModel

# Compute coherence for the 5-topic model
coherence_model = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v')

coherence_score = coherence_model.get_coherence()
print(f"Coherence score (5 topics): {coherence_score:.4f}")

### 📝 Activity 2a: Coherence Elbow Plot

Train LDA models with different numbers of topics (2, 5, 8, 11, 14, 17, 20) and compute the coherence score for each. Then plot the results to find the "elbow" — the point where adding more topics stops helping.

Steps:
1. Loop through `topic_range = range(2, 21, 3)` (gives you 2, 5, 8, 11, 14, 17, 20)
2. For each number, train an LDA model and compute its coherence score
3. Plot number of topics (x) vs. coherence score (y)

**Note:** This will take 1–2 minutes to run. Be patient!

*Hint: Follow the pattern from the coherence demo above, but inside a loop.*

In [ ]:
# Activity 2a: Build an elbow plot of coherence vs number of topics.
# Expected output: a printed table of (k, coherence) and a line plot.
# Remember from class: the *elbow* is where the rise levels off -- it isn't
# necessarily the peak. Pick the simplest model that captures most of the gain.

# TODO:
#  1. For each k in range(2, 21, 3): train an LDA model, compute coherence
#     with CoherenceModel(..., coherence='c_v').get_coherence(), append to a list
#  2. Print each (k, score)
#  3. Plot the scores with plt.plot(..., marker='o')
k_values = list(range(2, 21, 3))
coherence_scores = []
for k in k_values:
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10
    )
    coherence_model = CoherenceModel(
        model=lda_model,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence_scores.append(coherence_model.get_coherence())
    print(f"Coherence score (k={k}): {coherence_scores[-1]:.4f}")

plt.plot(k_values, coherence_scores, marker='o')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score')
plt.title('Coherence Score vs. Number of Topics')
plt.show()


### 📝 Activity 2b: Visualize with pyLDAvis

Use `pyLDAvis` to create an interactive visualization of your 8-topic model (or whichever model you prefer from the elbow plot).

Steps:
1. Import `pyLDAvis.gensim_models` and `pyLDAvis`
2. Call `pyLDAvis.enable_notebook()`
3. Use `gensimvis.prepare()` with your model, corpus, and dictionary
4. Display the visualization

**How to explore the visualization:**
- Click on circles (topics) in the left panel to see their top words on the right
- Bigger circles = topics that cover more documents
- Topics far apart are more distinct; overlapping circles share many words
- Adjust the relevance slider (λ) to see words unique to a topic (left) vs. most frequent (right)

In [ ]:
# Activity 2b: Visualize the 8-topic model with pyLDAvis.
# Expected output: an interactive HTML visualization with circles on the left
# (one per topic) and a top-words panel on the right.

# TODO:
#  1. Import pyLDAvis and pyLDAvis.gensim_models as gensimvis
#  2. Enable notebook mode
#  3. Build the visualization with gensimvis.prepare(model, corpus, dictionary)
#  4. Display it

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()
vis = gensimvis.prepare(lda_model_8, corpus, dictionary)
vis

### Demo: Topic Coverage Across the Corpus

**Coverage** tells you what fraction of the corpus each topic represents on average. A topic with 25% coverage means that, across all documents, about a quarter of the content is about that topic.

In [ ]:
# Compute topic coverage for the 5-topic model
topic_coverage = np.zeros(lda_model.num_topics)

for doc_bow in corpus:
    for topic_id, proportion in lda_model.get_document_topics(
            doc_bow, minimum_probability=0):
        topic_coverage[topic_id] += proportion

# Average across all documents
topic_coverage /= len(corpus)

print("Topic coverage (5-topic model):")
for i, cov in enumerate(topic_coverage):
    print(f"  Topic {i}: {cov*100:.1f}%")

### Demo: Comparing Topics Across Newspapers

Different newspapers may cover different topics. Let's compute topic coverage **per newspaper** to see how they differ.

In [ ]:
# See which newspapers are in our sample
print("Articles per newspaper:")
print(df['newspaper_name'].value_counts())

In [ ]:
# Compute topic coverage per newspaper (using the 5-topic model)
newspapers = df['newspaper_name'].unique()

# Group article indices by newspaper
indices_by_newspaper = {}
for newspaper in newspapers:
    indices_by_newspaper[newspaper] = df[df['newspaper_name'] == newspaper].index.tolist()

# Compute coverage for each newspaper
topic_coverage_by_newspaper = {}
for newspaper, indices in indices_by_newspaper.items():
    coverage = np.zeros(lda_model.num_topics)
    for idx in indices:
        for topic_id, proportion in lda_model.get_document_topics(
                corpus[idx], minimum_probability=0):
            coverage[topic_id] += proportion
    coverage /= len(indices)  # average
    topic_coverage_by_newspaper[newspaper] = coverage

# Display as a DataFrame
coverage_df = pd.DataFrame.from_dict(
    topic_coverage_by_newspaper, orient='index',
    columns=[f'Topic {i}' for i in range(lda_model.num_topics)])

print("Topic coverage by newspaper (5-topic model):")
coverage_df.round(3)

### 📝 Activity 2c: Compare Two Newspapers

Pick two newspapers from the dataset and compare their topic coverage using your **8-topic model** (`lda_model_8`).

Steps:
1. Compute topic coverage for the 8-topic model, grouped by newspaper (follow the demo pattern above)
2. Pick two newspapers (e.g., the one with the most articles and one with fewer)
3. Print their topic coverage side by side

After completing, answer: **What topics differ most between the two newspapers?**

*Hint: Follow the demo pattern above, but use `lda_model_8` instead of `lda_model`.*

In [ ]:
# Activity 2c: Compare topic coverage across two newspapers using lda_model_8.
# Expected output: two side-by-side breakdowns showing topic % per newspaper.
# This will reveal which themes each paper covers more / less.

# TODO:
#  1. Build a per-newspaper coverage vector by looping through indices_by_newspaper
#     and averaging get_document_topics(...) over each newspaper's docs
#  2. Wrap the result in a pandas DataFrame (rows = newspapers, cols = topics)
#  3. Pick two newspapers (e.g., the first two) and print their topic % side by side

newspaper_topic_data = {}
for newspaper, indices in indices_by_newspaper.items():
    coverage = np.zeros(lda_model_8.num_topics)
    for idx in indices:
        for topic_id, proportion in lda_model_8.get_document_topics(
                corpus[idx], minimum_probability=0):
            coverage[topic_id] += proportion
    coverage /= len(indices)  # average
    newspaper_topic_data[newspaper] = coverage

coverage_8_df = pd.DataFrame.from_dict(
    newspaper_topic_data, orient='index',
    columns=[f'Topic {i}' for i in range(lda_model_8.num_topics)])
topic_coverage_df = pd.DataFrame(newspaper_topic_data).T
topic_coverage_df.columns = [f"Topic {i}" for i in range(8)]
paper_names = list(indices_by_newspaper.keys())[:2]
comparison_df = topic_coverage_df.loc[paper_names].T
print(f"Side-by-Side Topic Coverage: {paper_names[0]} vs {paper_names[1]}")
print("-" * 60)
print((comparison_df * 100).round(2).astype(str) + '%')



---
# 🏠 AFTER-CLASS ACTIVITIES

Complete these on your own after lecture (~30 min total).

---

## Activity 3: Single-Document Topic Analysis (~15 min)

In Activity 1c you looked at one document. Now explore several documents to build intuition for how LDA assigns topics.

### 📝 Activity 3a: Find a Document Dominated by One Topic

Loop through the first 50 documents and find one where a **single topic accounts for 80% or more** of the document.

Steps:
1. Loop through `range(50)`
2. For each document, get its topic distribution with `lda_model_8.get_document_topics(corpus[i])`
3. Check if any topic has proportion > 0.80
4. Print the document index, dominant topic, and the first 200 characters of text

*Hint: `max(doc_topics, key=lambda x: x[1])` gives you the topic with the highest proportion.*

In [ ]:
for i in range(50):
    doc_topics = lda_model_8.get_document_topics(corpus[i])
    max_topic, max_proportion = max(doc_topics, key=lambda x: x[1])
    if max_proportion > 0.80:
        label = topic_labels.get(max_topic, "No Label")
        snippet = df.loc[i, 'article'][:200].replace('\n', ' ')
        print(f"Document Index: {i}")
        print(f"Dominant Topic: {max_topic} ({label})")
        print(f"Concentration:  {max_proportion*100:.2f}%")
        print(f"Snippet:        {snippet}...")
        print("-" * 40)

### 📝 Activity 3b: Find a Document with Mixed Topics

Now find a document where **no single topic exceeds 40%** — meaning it's a true mixture of several themes.

Steps:
1. Loop through the first 100 documents
2. For each, check if the max topic proportion is < 0.40
3. Print the full topic breakdown and the first 300 characters of text

In [ ]:
# Activity 3b: Find a document where NO topic exceeds 40% -- a true mixture.
# Expected output: the full topic breakdown for one mixed-theme doc plus a
# ~300-char snippet so you can see why several topics fit.

# TODO:
#  1. Loop over the first 100 docs
#  2. For each, find the maximum topic proportion
#  3. If it's < 0.40, print the breakdown and the article snippet, then break
for i in range(100):
    doc_topics = lda_model_8.get_document_topics(corpus[i])
    max_proportion = max(doc_topics, key=lambda x: x[1])[1]
    if max_proportion < 0.40:
        print



---
## Activity 4: Topic Coverage Bar Chart (~15 min)

Create a **grouped bar chart** comparing topic coverage across two newspapers. This is a great way to visually communicate how different sources emphasize different themes.

### 📝 Activity 4a: Grouped Bar Chart

Using the topic coverage data from Activity 2c, create a grouped bar chart with:
- X-axis: Topic numbers (0 through 7)
- Y-axis: Coverage percentage
- Two bars per topic: one for each newspaper
- A legend identifying which bar is which newspaper

Steps:
1. Use `np.arange(8)` for x positions
2. Use `plt.bar(x - width/2, ...)` for the first newspaper
3. Use `plt.bar(x + width/2, ...)` for the second newspaper
4. Add labels, title, and legend

*Hint: Look back at the grouped bar chart demo from Lecture 6.*

In [ ]:
# Activity 4a: Build a grouped bar chart comparing two newspapers across the 8 topics.
# Expected output: 8 pairs of bars, with a legend, axis labels, and title.

# TODO:
#  1. Pick paper1 and paper2 from coverage_8_df (e.g., the first two indices)
#  2. Use np.arange(8) for x positions and width = 0.35
#  3. Plot ax.bar(x - width/2, paper1 values * 100, width=width, label=paper1)
#     and ax.bar(x + width/2, paper2 values * 100, width=width, label=paper2)
#  4. Set xticks, xlabel, ylabel, title, and legend
paper1 = coverage_8_df.iloc[0]
paper2 = coverage_8_df.iloc[1]
paper1_name = coverage_8_df.index[0]
paper2_name = coverage_8_df.index[1]
x = np.arange(8)
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, paper1 * 100, width=width, label=paper1_name)
ax.bar(x + width/2, paper2 * 100, width=width, label=paper2_name)
ax.set_xticks(x)
ax.set_xlabel("Topics")
ax.set_ylabel("Topic Coverage (%)")
ax.set_title("Topic Distribution Comparison Across Two Newspapers")
ax.legend()
plt.show()




### 📝 Activity 4b: Interpret the Chart

Based on your grouped bar chart, answer these questions in the cell below:

1. Which topic does Newspaper 1 cover more than Newspaper 2?
2. Which topic does Newspaper 2 cover more?
3. Are there any topics where both newspapers have similar coverage?
4. Do the differences make sense given what you know about the topic words?

In [ ]:
# Activity 4b: Write your interpretation as comments below.
# This is reflective writing -- there is no single right answer.

# 1. Newspaper 1 covers Topic _7__ more because: __It is more relevant for Evening Star.____________
# 2. Newspaper 2 covers Topic ___3 more because: __It covers more things relevent to Chapel Hill_____________
# 3. Both newspapers have similar coverage for Topic 5___
# 4. This makes sense because: _They are both similar newspapers, covering similar topics but on a different scale.___________


---
## Summary

Here's what you practiced in this notebook:

| Activity | Topic | Key Methods |
|----------|-------|-------------|
| 1a | Train LDA with 8 topics | `LdaModel()`, `print_topics()` |
| 1b | Label topics | Human interpretation of word lists |
| 1c | Single document topics | `get_document_topics()` |
| 2a | Coherence elbow plot | `CoherenceModel()`, `plt.plot()` |
| 2b | pyLDAvis visualization | `gensimvis.prepare()`, `pyLDAvis.display()` |
| 2c | Compare newspapers | Topic coverage by group |
| 3a | Dominant-topic documents | Filtering by proportion |
| 3b | Mixed-topic documents | Filtering by max proportion |
| 4a | Grouped bar chart | `plt.bar()` with offsets |
| 4b | Interpret results | Written analysis |

**Key takeaway:** Topic modeling lets you discover themes in text data without any labels. The workflow is: clean text → build dictionary & corpus → train LDA → interpret and tune.

---
## Reminders

- **Submit** this notebook via Canvas by the deadline.
- **Office hours**: Check the syllabus for times and location.
- **Project #2**: Make sure you are making progress on your unstructured data project. Reach out if you have questions!